# 03 – Person Details

Esplorazione e data cleaning del dataset `person_details.csv`.

**Colonne:**

| Colonna | Descrizione |
|---|---|
| `person_mal_id` | ID univoco della persona su MyAnimeList|
| `url` | URL della pagina MAL della persona |
| `website_url` | Sito web personale |
| `image_url` | URL dell'immagine del profilo |
| `name` | Nome completo |
| `given_name` | Nome proprio |
| `family_name` | Cognome |
| `birthday` | Data di nascita |
| `favorites` | Numero di utenti che l'hanno tra i preferiti |
| `relevant_location` | Posizione |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [ ]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
df_pd = pd.read_csv('../datasets/person_details.csv')
print(f'Shape: {df_pd.shape}')
print()
df_pd.info()
df_pd.head()

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [1]:
n_originale = len(df_pd)

mask_dup = df_pd.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_pd[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_pd.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_pd):,}')

NameError: name 'df_pd' is not defined

## 2. Analisi colonna per colonna

### 2.1 `person_mal_id`

ID univoco della persona su MAL. È la **chiave primaria** del dataset. I valori duplicati **non sono attesi**.

In [ ]:
analyze(df_pd['person_mal_id'])

**Osservazioni:**
- Nessun null
- Il dtype è già `int64` quindi nessuna conversione necessaria.
- Si nota che 99.87% dei valori sono univoci. Stampiamo un campione per verificare i duplicati.

In [ ]:
print(f'Null in person_mal_id      : {df_pd["person_mal_id"].isna().sum()}')
print(f'Duplicati in person_mal_id : {df_pd["person_mal_id"].duplicated().sum()}')

# Campione righe duplicate
mask_pk_dup = df_pd.duplicated(subset=['person_mal_id'], keep=False)
df_dup = df_pd[mask_pk_dup].sort_values('person_mal_id')
print(f'\nRighe coinvolte: {len(df_dup)} | person_mal_id unici duplicati: {df_dup["person_mal_id"].nunique()}')
df_dup.head(20)

I duplicati consistono in righe in cui l'unico valore che cambia è `relevant_location`. Non possiamo rimuovere le righe in quanto non è possibile sapere quale location è quella corretta. In più, la presenza di questa colonna rompe la struttura della tabella dove ogni persona dovrebbe comparire in una sola riga dove la chiave primaria è   `person_mal_id`. Abbiamo deciso di rimuovere la colonna `relevant_location` e infine rimuovere le righe duplicate mantenendo solo la prima occorrenza.

In [ ]:
# Rimozione colonna
df_pd.drop(columns=['relevant_location'], inplace=True)

# Rimozione duplicati indotti (ora sono duplicati esatti)
n_prima = len(df_pd)
df_pd.drop_duplicates(subset=['person_mal_id'], keep='first', inplace=True)
df_pd.reset_index(drop=True, inplace=True)
print(f'Righe prima : {n_prima:,}')
print(f'Righe dopo  : {len(df_pd):,}')
print(f'Rimosse     : {n_prima - len(df_pd):,}')

### 2.2 `url`

URL della pagina MAL della persona. Ci aspettiamo unicità e nessun null. Utilizziamo `strip()` per rimuovere eventuali spazzi vuoti all inizio e fine del URL. Effettuiamo poi l'analisi usando la nostra libreria `dataset_analyzer` e facciamo un controllo sul format del URL. Controlliamo anche la consistenza `person_mal_id` ↔ `url`. L'URL contiene l'ID del personaggio nel percorso (`/people/{id}/`). Verifichiamo che l'ID estratto dall'URL corrisponda al valore di `person_mal_id` per ogni riga.

In [ ]:
df_pd['url'] = df_pd['url'].str.strip()
analyze(df_pd['url'])

pattern = r'^https://myanimelist\.net/people/\d+/\S+$'
non_conformi = df_pd[~df_pd['url'].str.match(pattern)]
print(f'URL non conformi al formato atteso: {len(non_conformi)}')
if len(non_conformi) > 0:
    print(non_conformi[['person_mal_id', 'url', 'name']].to_string(index=True))

id_from_url = df_pd['url'].str.extract(r'/people/(\d+)/')[0].astype(int)
mismatch = (id_from_url != df_pd['person_mal_id'])
print(f'\nRighe con ID non coerente tra url e person_mal_id: {mismatch.sum()}')

**Nessuna pulizia necessaria.**
Nessun null, tutti gli URL sono unici e rispettano il pattern atteso (coerente con `person_mal_id` come chiave primaria).

### 2.3 `website_url`

Sito web personale della persona, campo **opzionale** su MAL. I null sono attesi in quanto alcune persone potrebbero non aver inserito il proprio sito. I duplicati sono possibili ma rari come per esempio più persone potrebbero condividere un sito.

In [ ]:
df_pd['website_url'] = df_pd['website_url'].str.strip()
analyze(df_pd['website_url'])

# 1. Pattern: i valori non-null devono iniziare con http:// o https://
mask_invalid = df_pd['website_url'].notna() & ~df_pd['website_url'].str.match(r'^https?://')
print(f'URL senza protocollo http/https : {mask_invalid.sum()}')
if mask_invalid.sum() > 0:
    print(df_pd.loc[mask_invalid, ['person_mal_id', 'name', 'website_url']].to_string(index=True))

# 2. Valori anomali: presenza di spazi o assenza di punto (probabile non-URL)
mask_anomali = (df_pd['website_url'].notna()
                & (df_pd['website_url'].str.contains(r' ') | ~df_pd['website_url'].str.contains(r'\.')))
print(f'Valori anomali (spazi o senza punto) : {mask_anomali.sum()}')
if mask_anomali.sum() > 0:
    print(df_pd.loc[mask_anomali, ['person_mal_id', 'name', 'website_url']].to_string(index=True))


- I null (59.186, ~77%) sono strutturali: la maggior parte delle persone non ha un sito web registrato su MAL.
- Risultano 22 righe con valori anomali che abbiamo deciso di tenere perché non sono rilevanti per la nostra analisi statistica e per non perdere le informazioni delle altre colonne.

**Nessuna pulizia necessaria.**

### 2.4 `image_url`

URL dell'immagine del profilo della persona su MAL. Effettuiamo l'analisi usando la nostra libreria `dataset_analyzer` dopo aver usato `strip()` per rimuovere spazi vuoti e facciamo poi un controllo sul format del URL.

In [ ]:
df_pd['image_url'] = df_pd['image_url'].str.strip()
analyze(df_pd['image_url'])

pattern_img = r'^https://cdn\.myanimelist\.net/(images/voiceactors/\d+/\d+\.jpg|img/sp/icon/apple-touch-icon-256\.png)$'
non_conformi_img = df_pd[~df_pd['image_url'].str.match(pattern_img)]
print(f'Immagini non conformi al formato atteso: {len(non_conformi_img)}')
if len(non_conformi_img) > 0:
    print(non_conformi_img[['person_mal_id', 'name', 'image_url']].to_string(index=True))

**Osservazioni:**
- Non ci sono stringhe vuote o null. Tutta la colonna è popolata.
- Ci sono 48.165 URL duplicati (62.88%). Tutti corrispondono a un link che porta a un'immagine placeholder. A ogni persona senza immagine reale viene assegnato il logo MAL. Non è un errore e non è necessaria nessuna pulizia.
- Ci sono 28.429 URL unici (37.12%). Tutti le persone con immagine reale hanno un URL univoco.
- Tutte le URL seguono il formato corretto senza nessuna anomalia strutturale.
- Nella distribuzione delle lunghezze, le fasce 59–63 hanno 0 occorrenze, il che significa che le URL si dividono in due gruppi: percorso /images/characters/ (55–58 caratteri) e percorso placeholder /img/sp/icon/ (64 caratteri).

**Nessuna pulizia necessaria**

### 2.5 `name`

Nome completo della persona. I duplicati sono possibili.

In [ ]:
df_pd['name'] = df_pd['name'].str.strip()
analyze(df_pd['name'])

**Osservazione:**
- Ci sono 2 valori null. Li stampiamo per vedere se ci sono altri dati che possono essere utili.
- Ci sono 2.084 duplicati. Stampiamo una parte per indagare meglio.
- Ci sono nomi con un solo carattere, contenenti @ oppure interamente numerici. Stampiamo una parte per verificare se si tratta di errori.

In [ ]:
# Null
null_name = df_pd[df_pd['name'].isna()]
print(f'Righe con name null: {len(null_name)}')
print(null_name.to_string(index=True))

# Duplicati
dup_name = df_pd[df_pd['name'].duplicated(keep=False)].sort_values('name')
print(f'\nRighe con name duplicato: {len(dup_name)}')
print(dup_name.head(20).to_string(index=True))

# Nomi da 1 carattere
singoli = df_pd[df_pd['name'].str.len() == 1]
print(f'\nNomi da 1 carattere: {len(singoli)}')
print(singoli.head(10).to_string(index=True))

# Nomi contenenti @
con_at = df_pd[df_pd['name'].str.contains('@|＠', regex=True, na=False)]
print(f"\nNomi contenenti '@': {len(con_at)}")
print(con_at.head(10).to_string(index=True))

# Nomi solo numerici
solo_num = df_pd[df_pd['name'].str.fullmatch(r'\d+', na=False)]
print(f'\nNomi solo numerici: {len(solo_num)}')
print(solo_num.head(10).to_string(index=True))

- I due valori nulli corrispondono a due profili vuoti che verranno rimossi.
- Il resto dei valori corrisponde a righe che contengono informazioni parzialmente mancanti. Prima di decidere se rimuoverli o meno, verifichiamo se compaiono in altri dataset come foreign keys. Le righe che non vengono mai referenziate, vengono rimosse.

In [ ]:
# Rimozione righe con name null
print(f'Righe prima: {len(df_pd):,}')
df_pd.dropna(subset=['name'], inplace=True)
df_pd.reset_index(drop=True, inplace=True)
print(f'Righe dopo : {len(df_pd):,}')

# Verifica referenze degli ID anomali restanti negli altri dataset
from foreign_key_analyzer import check_pk_referenced

ids_anomali = pd.Series(
    list(
        set(singoli['person_mal_id'])
        | set(con_at['person_mal_id'])
        | set(solo_num['person_mal_id'])
        | set(dup_name['person_mal_id'])
    ),
    name='person_mal_id'
)

df_paw  = pd.read_csv('../datasets/person_anime_works.csv')
df_pvw  = pd.read_csv('../datasets/person_voice_works.csv')
df_pan  = pd.read_csv('../datasets/person_alternate_names.csv')
df_favs = pd.read_csv('../datasets/favs.csv')

mask_unreferenced = check_pk_referenced(
    parent   = ids_anomali,
    children = {
        'person_anime_works'     : df_paw['person_mal_id'],
        'person_voice_works'     : df_pvw['person_mal_id'],
        'person_alternate_names' : df_pan['person_mal_id'],
        'favs (people)'          : df_favs[df_favs['fav_type'] == 'people']['id'],
    },
    parent_df   = df_pd[df_pd['person_mal_id'].isin(ids_anomali)],
    sample_rows = 10
)
# Rimozione degli ID anomali non referenziati in nessuna tabella figlia
ids_da_rimuovere = ids_anomali[mask_unreferenced]
print(f'Righe prima della rimozione: {len(df_pd):,}')
df_pd = df_pd[~df_pd['person_mal_id'].isin(ids_da_rimuovere)]
df_pd.reset_index(drop=True, inplace=True)
print(f'Righe dopo la rimozione    : {len(df_pd):,}')
print(f'Righe rimosse              : {len(ids_da_rimuovere):,}')

### 2.6 `given_name`

Nome proprio in giapponese. I null sono strutturali: non disponibile per persone non giapponesi o quando non registrato su MAL.

In [ ]:
df_pd['given_name'] = df_pd['given_name'].str.strip()
analyze(df_pd['given_name'])

**Osservazioni:**
- I null (~39%) sono strutturali e non richiedono intervento.
- Dalla distribuzione delle lunghezze (min 1, max 28, media ~2.18) emergono valori anomali:
  - stringhe con soli caratteri latini (il campo dovrebbe contenere kanji/kana);
  - stringhe molto lunghe (> 15 car.) che non sembrano nomi propri;
  - valori con parentesi che mescolano lingue o aggiungono annotazioni.

Verifichiamo e puliamo.

In [ ]:
# Stringhe con soli caratteri latini (attesi kanji/kana)
solo_latin_gn = df_pd[df_pd['given_name'].str.fullmatch(r'[A-Za-z\s\-\.]+', na=False)]
print(f'Solo caratteri latini: {len(solo_latin_gn)}')
print(solo_latin_gn[['person_mal_id', 'name', 'given_name']].head(15).to_string(index=True))

# Stringhe molto lunghe (> 15 caratteri)
lunghi_gn = df_pd[df_pd['given_name'].str.len() > 15]
print(f'Lunghezza > 15 caratteri: {len(lunghi_gn)}')
print(lunghi_gn[['person_mal_id', 'name', 'given_name']].head(15).to_string(index=True))

# Valori contenenti parentesi
con_par_gn = df_pd[df_pd['given_name'].str.contains(r'[\(\)\[\]]', na=False)]
print(f'Contenenti parentesi: {len(con_par_gn)}')
print(con_par_gn[['person_mal_id', 'name', 'given_name']].head(15).to_string(index=True))

- Le stringhe con soli caratteri latini e quelle molto lunghe sono profili non giapponesi o voci di fantasia. Il campo `given_name` non è significativo per loro e quindi lo impostiamo a `null`.
- I valori con parentesi rientrano nelle categorie precedenti (es. trascrizioni multi-lingua), trattati nello stesso modo.

In [ ]:
# Maschera valori anomali in given_name
mask_anomali_gn = (
    df_pd['given_name'].str.fullmatch(r'[A-Za-z\s\-\.]+', na=False)   # solo Latin
    | (df_pd['given_name'].str.len() > 15)                                # troppo lunghi
    | df_pd['given_name'].str.contains(r'[\(\)\[\]]', na=False)       # parentesi
)

n_anomali = mask_anomali_gn.sum()
print(f'Valori anomali in given_name: {n_anomali}')

# Azzeriamo a NaN anziché rimuovere la riga: la persona può essere valida
df_pd.loc[mask_anomali_gn, 'given_name'] = np.nan
print(f'Null in given_name dopo pulizia: {df_pd["given_name"].isna().sum():,} ({df_pd["given_name"].isna().mean()*100:.1f}%)')

### 2.7 `family_name`

Cognome in giapponese (kanji/kana). Analogo a `given_name`, i null sono strutturali per le stesse ragioni.

In [ ]:
df_pd['family_name'] = df_pd['family_name'].str.strip()
analyze(df_pd['family_name'])

**Osservazioni:**
- I null (~24%) sono strutturali e non richiedono intervento.
- Dalla distribuzione delle lunghezze (min 1, max 31, media ~2.37) emergono valori anomali:
  - stringhe con soli caratteri latini (il campo dovrebbe contenere kanji/kana);
  - stringhe molto lunghe (> 15 car.) con nomi di band o valori compositi;
  - valori con parentesi che mescolano lingue o aggiungono annotazioni.
Verifichiamo e puliamo.

In [ ]:
# Stringhe con soli caratteri latini (attesi kanji/kana)
solo_latin_fn = df_pd[df_pd['family_name'].str.fullmatch(r'[A-Za-z\s\-\.]+', na=False)]
print(f'Solo caratteri latini: {len(solo_latin_fn)}')
print(solo_latin_fn[['person_mal_id', 'name', 'family_name']].head(15).to_string(index=True))

# Stringhe molto lunghe (> 15 caratteri)
lunghi_fn = df_pd[df_pd['family_name'].str.len() > 15]
print(f'Lunghezza > 15 caratteri: {len(lunghi_fn)}')
print(lunghi_fn[['person_mal_id', 'name', 'family_name']].head(15).to_string(index=True))

# Valori contenenti parentesi
con_par_fn = df_pd[df_pd['family_name'].str.contains(r'[\(\)\[\]]', na=False)]
print(f'Contenenti parentesi: {len(con_par_fn)}')
print(con_par_fn[['person_mal_id', 'name', 'family_name']].head(15).to_string(index=True))

**Osservazioni:**
- Le stringhe con soli caratteri latini e quelle oltre i 15 caratteri non sono cognomi giapponesi validi quindi li impostiamo a `null`.
- I valori con parentesi rientrano nelle stesse categorie (valori compositi o multi-lingua), trattati nello stesso modo.

In [ ]:
# Maschera valori anomali in family_name
mask_anomali_fn = (
    df_pd['family_name'].str.fullmatch(r'[A-Za-z\s\-\.]+', na=False)   # solo Latin
    | (df_pd['family_name'].str.len() > 15)                              # troppo lunghi
    | df_pd['family_name'].str.contains(r'[\(\)\[\]]', na=False)        # parentesi
)

n_anomali_fn = mask_anomali_fn.sum()
print(f'Valori anomali in family_name: {n_anomali_fn}')

df_pd.loc[mask_anomali_fn, 'family_name'] = np.nan
print(f'Null in family_name dopo pulizia: {df_pd["family_name"].isna().sum():,} ({df_pd["family_name"].isna().mean()*100:.1f}%)')

### 2.8 `birthday`

Data di nascita come stringa ISO 8601 con timezone UTC, campo **opzionale**. I null sono strutturali: molte persone non rendono pubblica la data di nascita su MAL. Richiede conversione a `datetime64` e verifica di date implausibili.

In [ ]:
df_pd['birthday'] = df_pd['birthday'].str.strip()
analyze(df_pd['birthday'])

**Osservazioni:**
- I valori sono stringhe che convertiamo a `datetime64`.
- I null (~78%) sono strutturali: date di nascita non pubbliche o non registrate su MAL.
- Controlliamo il range per verificare possibili date errate che verranno impostate a `null`.

In [ ]:
df_pd['birthday'] = pd.to_datetime(df_pd['birthday'], errors='coerce', utc=True)
print(f'Range          : {df_pd["birthday"].min()} → {df_pd["birthday"].max()}')
print()

# Date implausibili: anno < 1800
mask_passato = df_pd['birthday'].dt.year < 1800
n_passato = mask_passato.sum()
print(f'Date con anno < 1800: {n_passato}')
print(df_pd.loc[mask_passato, ['person_mal_id', 'name', 'url', 'birthday']].head(30).to_string(index=True))

# Date recenti
mask_futuro = df_pd['birthday'].dt.year > 2020
n_futuro = mask_futuro.sum()
print(f'\nDate recenti: {n_futuro}')
print(df_pd.loc[mask_futuro, ['person_mal_id', 'name', 'url', 'birthday']].head(30).to_string(index=True))

# Rimozione di tutti i birthday il cui anno non inizia con 1 o 2 (fuori dal range 1000-2026)
mask_bad_date = df_pd['birthday'].notna() & ~df_pd['birthday'].dt.year.between(1000, 2026)
df_pd.loc[mask_bad_date, 'birthday'] = pd.NaT
print(f'\nDate anomale impostate a NaT : {mask_bad_date.sum()}')
print(f'birthday dtype : {df_pd["birthday"].dtype}')
print(f'Null residui   : {df_pd["birthday"].isna().sum():,}')

Abbiamo effettuato un'analisi delle date estreme fuori dal range 1800 - 2020. Dalle date emerse prima del 1800, si nota che si tratta principalmente di autori o compositori classici mentre i valori dopo il 2020 corrispondono ad AI oppure gruppi musicali formati recentemente. Abbiamo impostato a null solo gli anni che corrispondevano a valori evidentemente errati come `0800`, `0447`, `0296`, `0178`, etc.